In [2]:
import numpy as np
import pandas as pd

# =====================================================================
# 1. LOAD & STANDARDIZE THE DATA
# =====================================================================
df = pd.read_csv(
    "C:/Users/adaml/OneDrive/Documents/Data Projects/Food_items_for_meal_engine.csv",
    encoding="cp1252",
)

df.columns = df.columns.str.strip()

df["Cost per 100g(£)"] = (
    df["Cost per 100g(£)"]
    .astype(str)
    .str.replace("£", "")
    .str.strip()
)
df["Cost per 100g(£)"] = pd.to_numeric(df["Cost per 100g(£)"], errors="coerce").fillna(0.0)


# =====================================================================
# 2. DEFINE MACRO TARGETS BY DAY TYPE
# =====================================================================
def get_daily_targets(day_type):
    target_calories = 2050.0
    if day_type == "Gym Day":
        return {
            "calories": target_calories,
            "protein": 154.0,
            "carbs": 256.0,
            "fats": 46.0,
        }
    elif day_type == "Running Day":
        return {
            "calories": target_calories,
            "protein": 103.0,
            "carbs": 282.0,
            "fats": 57.0,
        }


# =====================================================================
# 3. HELPER FUNCTION: QUANTITY AND MACRO METRIC SCALAR
# =====================================================================
def calculate_item_macros(row, quantity=1.0):
    scalar = (row["Serving_size_g"] / 100.0) * quantity
    return {
        "name": row["Name"],
        "unit_type": row["Unit_type"],
        "serving_size_g": row["Serving_size_g"],
        "quantity": quantity,
        "calories": row["Calories_per_100g"] * scalar,
        "protein": row["Protein_g"] * scalar,
        "fats": row["Fat_g"] * scalar,
        "carbs": row["Carbs_g"] * scalar,
        "cost": row["Cost per 100g(£)"] * scalar,
        "prep_time": row["Prep_time(mins)"],
        "raw_row": row,
    }


# =====================================================================
# 4. SINGLE-DAY GENERATION ENGINE (WITH VARIETY FILTER)
# =====================================================================
def generate_daily_menu(df, day_type, excluded_item_names=[], max_available_prep_time=160):
    targets = get_daily_targets(day_type)
    p_tol, c_tol, f_tol = 0.05, 0.05, 0.05

    # Filter out recently used main proteins/starches to enforce weekly variety
    available_df = df[~df["Name"].isin(excluded_item_names)]
    if available_df.empty:
        available_df = df  # Fallback if pool gets exhausted

    attempts = 0
    while attempts < 5000:
        attempts += 1
        daily_menu = []

        current_p_tol = p_tol + (attempts // 100 * 0.002)
        current_c_tol = c_tol + (attempts // 100 * 0.002)
        current_f_tol = f_tol + (attempts // 100 * 0.002)

        # --- 1. EXTRACT STRUCTURAL MEAL SUB-POOLS FROM AVAILABLE DF ---
        b_pool = available_df[available_df["Meal_type"].str.contains("Breakfast", case=False, na=False)]
        b_savory = b_pool[b_pool["Category_type"] == "Savory"]
        b_starch = b_pool[b_pool["Category_type"] == "Starch"]
        b_sweet = b_pool[b_pool["Category_type"].isin(["Fruit", "Dessert", "Supplement"])]

        ld_pool = available_df[available_df["Meal_type"].str.contains("Lunch|Dinner", case=False, na=False)]
        ld_savory = ld_pool[ld_pool["Category_type"] == "Savory"]
        ld_starch = ld_pool[ld_pool["Category_type"] == "Starch"]
        ld_veg = ld_pool[ld_pool["Category_type"] == "Vegetable"]

        all_starches = available_df[available_df["Category_type"] == "Starch"]
        all_savories = available_df[available_df["Category_type"] == "Savory"]
        all_sweets = available_df[available_df["Category_type"].isin(["Fruit", "Dessert", "Supplement", "Snack"])]

        if all_starches.empty or all_savories.empty or ld_veg.empty:
            all_starches = df[df["Category_type"] == "Starch"]
            all_savories = df[df["Category_type"] == "Savory"]

        # --- 2. DYNAMIC SELECTION SEEDING ---
        b_sav_item = b_savory.sample().iloc[0] if not b_savory.empty else all_savories.sample().iloc[0]
        b_sta_item = b_starch.sample().iloc[0] if not b_starch.empty else all_starches.sample().iloc[0]
        b_sw_item = b_sweet.sample().iloc[0] if not b_sweet.empty else all_sweets.sample().iloc[0]

        l_sav_item = ld_savory.sample().iloc[0] if not ld_savory.empty else all_savories.sample().iloc[0]
        l_sta_item = ld_starch.sample().iloc[0] if not ld_starch.empty else all_starches.sample().iloc[0]
        l_veg_item = ld_veg.sample().iloc[0]

        d_sav_item = ld_savory.sample().iloc[0] if not ld_savory.empty else all_savories.sample().iloc[0]
        d_sta_item = ld_starch.sample().iloc[0] if not ld_starch.empty else all_starches.sample().iloc[0]
        d_veg_item = ld_veg.sample().iloc[0]

        daily_menu.append(calculate_item_macros(b_sav_item))
        daily_menu.append(calculate_item_macros(b_sta_item))
        daily_menu.append(calculate_item_macros(b_sw_item))
        
        daily_menu.append(calculate_item_macros(l_sav_item))
        daily_menu.append(calculate_item_macros(l_sta_item))
        daily_menu.append(calculate_item_macros(l_veg_item))
        
        daily_menu.append(calculate_item_macros(d_sav_item))
        daily_menu.append(calculate_item_macros(d_sta_item))
        daily_menu.append(calculate_item_macros(d_veg_item))

        # =====================================================================
        # GLOBAL PORTION FAT COMPRESSION & SCALING
        # =====================================================================
        initial_fats = sum(item["fats"] for item in daily_menu)
        fat_safety_ceiling = targets["fats"] * 0.90

        if initial_fats > fat_safety_ceiling and initial_fats > 0:
            universal_downscaler = fat_safety_ceiling / initial_fats
            for item in daily_menu:
                item["quantity"] *= universal_downscaler

        # Protein Scaling
        temp_menu = [calculate_item_macros(item["raw_row"], quantity=item["quantity"]) for item in daily_menu]
        base_p = sum(item["protein"] for item in temp_menu)
        p_gap = targets["protein"] - base_p

        if p_gap > 0 and base_p > 0:
            clean_protein_items = [i for i in daily_menu if i["protein"] > (i["fats"] * 1.2) and i["protein"] > 2]
            if clean_protein_items:
                p_pool_total = sum(i["protein"] for i in clean_protein_items)
                if p_pool_total > 0:
                    p_multiplier = 1.0 + (p_gap / p_pool_total)
                    for item in clean_protein_items:
                        item["quantity"] *= min(p_multiplier, 3.5)

        # Carbohydrate Scaling
        temp_menu = [calculate_item_macros(item["raw_row"], quantity=item["quantity"]) for item in daily_menu]
        base_c = sum(item["carbs"] for item in temp_menu)
        c_gap = targets["carbs"] - base_c

        if c_gap > 0 and base_c > 0:
            clean_carb_items = [i for i in daily_menu if i["carbs"] > (i["fats"] * 2.5) and i["carbs"] > 8]
            if clean_carb_items:
                c_pool_total = sum(i["carbs"] for i in clean_carb_items)
                if c_pool_total > 0:
                    c_multiplier = 1.0 + (c_gap / c_pool_total)
                    for item in clean_carb_items:
                        item["quantity"] *= min(c_multiplier, 4.0)

        # Final re-computation & validation
        finalized_menu = [calculate_item_macros(item["raw_row"], quantity=item["quantity"]) for item in daily_menu]

        total_p = sum(item["protein"] for item in finalized_menu)
        total_c = sum(item["carbs"] for item in finalized_menu)
        total_f = sum(item["fats"] for item in finalized_menu)
        total_cost = sum(item["cost"] for item in finalized_menu)
        total_cal = (total_p * 4) + (total_c * 4) + (total_f * 9)

        #£80 budget per week constraint
        if total_cost > 11.42:
            continue

        p_pass = targets["protein"] * (1 - current_p_tol) <= total_p <= targets["protein"] * (1 + current_p_tol)
        c_pass = targets["carbs"] * (1 - current_c_tol) <= total_c <= targets["carbs"] * (1 + current_c_tol)
        f_pass = targets["fats"] * (1 - current_f_tol) <= total_f <= targets["fats"] * (1 + current_f_tol)

        if p_pass and c_pass and f_pass:
            return finalized_menu, total_cal, total_p, total_c, total_f, total_cost

    return None, 0, 0, 0, 0, 0


# =====================================================================
# 5. WEEKLY GENERATOR FUNCTION (7-DAY WRAPPER)
# =====================================================================
def generate_weekly_plan(df):
    days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    weekly_plan = {}
    total_weekly_spend = 0.0

    print("\n" + "="*70)
    print("  GENERATING 7-DAY OPTIMIZED MEAL PLAN...")
    print("="*70)

    for day in days_of_week:
        # Alternate schedule: Gym days vs Running days
        day_type = "Running Day" if day in ["Wednesday", "Saturday"] else "Gym Day"
        
        # Track main items used recently to ensure variety across the week
        recent_items = []
        menu, cal, p, c, f, cost = generate_daily_menu(df, day_type=day_type, excluded_item_names=recent_items)

        if menu:
            weekly_plan[day] = {
                "menu": menu, "calories": cal, "protein": p, 
                "carbs": c, "fats": f, "cost": cost, "day_type": day_type
            }
            total_weekly_spend += cost
            
            # Print daily view
            print(f"\n {day.upper()} ({day_type}) | Target Spend: £{cost:.2f} | {cal:.0f} kcal")
            print(f"   Macros -> P: {p:.1f}g | C: {c:.1f}g | F: {f:.1f}g")
            
            # Print Segmented Meals
            print("   - Breakfast:", ", ".join([f"{i['quantity']:.1f}x {i['name']}" for i in menu[0:3]]))
            print("   - Lunch:    ", ", ".join([f"{i['quantity']:.1f}x {i['name']}" for i in menu[3:6]]))
            print("   - Dinner:   ", ", ".join([f"{i['quantity']:.1f}x {i['name']}" for i in menu[6:9]]))
        else:
            print(f"\n❌ Could not resolve menu for {day}.")

    print("\n" + "="*70)
    print(f"  WEEKLY PLAN COMPLETE! Total Estimated Grocery Cost: £{total_weekly_spend:.2f}")
    print("="*70)

# =====================================================================
# 6. EXECUTE THE WEEKLY PLANNER
# =====================================================================
generate_weekly_plan(df)


 🗓️ GENERATING 7-DAY OPTIMIZED MEAL PLAN...

 MONDAY (Gym Day) | Target Spend: £7.86 | 2045 kcal
   Macros -> P: 154.3g | C: 257.0g | F: 44.4g
   - Breakfast: 1.9x Egg Whites, 1.9x NYB Bagels (plain), 1.0x RANONG Matcha Latte
   - Lunch:     1.9x Costco Chicken Fillet , 1.9x White Rice, 1.0x Cucumber
   - Dinner:    1.9x Beef flank, 1.9x White Rice, 1.9x Broccoli

 TUESDAY (Gym Day) | Target Spend: £8.24 | 2103 kcal
   Macros -> P: 157.5g | C: 267.6g | F: 44.7g
   - Breakfast: 2.2x Egg Whites, 2.5x NYB Bagels (plain), 1.1x Banana
   - Lunch:     2.2x Chicken Breast, 1.0x Indomie Mi Goreng, 2.2x Broccoli
   - Dinner:    2.2x Chicken Drumstick(S), 2.5x White Rice, 1.0x Cucumber

 WEDNESDAY (Running Day) | Target Spend: £9.74 | 2052 kcal
   Macros -> P: 105.1g | C: 273.6g | F: 59.7g
   - Breakfast: 1.2x Egg Whites, 3.7x NYB Bagels (plain), 1.9x Banana
   - Lunch:     1.2x Fish, 0.6x Indomie Mi Goreng, 0.6x Avocado
   - Dinner:    1.2x Fish, 0.6x Indomie Mi Goreng, 0.6x Avocado

 THURSDAY